# Lab 2: Predictive Analytics with Machine Learning

**Duration:** 2 weeks [18 Jun - 25 Jun, 2026]
**Due Date:** 25th June, 2026
**Format:** Jupyter Notebook / Google Colab
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In this lab you will run a complete machine-learning workflow on **two real tabular datasets**,
covering both **supervised** and **unsupervised** learning:

| # | Task | Dataset | Type | Target |
|---|------|---------|------|--------|
| 1 | **Regression** | NYC Yellow Taxi trips | Supervised | `tip_amount` (continuous) |
| 2 | **Multi-class classification** | Obesity-level prediction | Supervised | `NObeyesdad` (7 classes) |
| 3 | **Clustering (K-Means)** | Obesity features (labels hidden) | Unsupervised | discover patient groups |

Along the way you will practise **NumPy, Pandas, and scikit-learn** to load and explore data,
clean and preprocess it, engineer features, split it into **train / validation / test** sets,
train models, **check for overfitting**, and discover hidden structure with clustering.

> **Note:** In this lab your *reasoning* is graded just as heavily as your *code*. Every section
> ends with a **Student Reasoning** box — fill it in with full sentences that justify your choices.

### Topics covered
Supervised learning (regression & classification), unsupervised learning (K-Means clustering),
feature engineering, train/validation/test splits, model evaluation, and overfitting.


---
### Part 0: Repository Setup *(done outside this notebook)*

1. Create a **public** repository named `lab-2-predictive-analytics` on GitHub/GitLab.
2. Clone it locally (or link it to Google Colab).
3. Save this notebook inside the repo as `lab_2_predictive_analytics.ipynb`.
4. Add a `requirements.txt` (provided with this lab) listing: `numpy pandas scikit-learn matplotlib seaborn`.
5. Commit and push your finished, fully-run notebook at the end.

**Local setup**
```bash
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate
pip install -r requirements.txt
jupyter lab
```

Open a new notebook, rename it, and run the cells below directly
(the datasets are loaded straight from their URLs, so no Drive mount is required).


In [2]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# You will import the specific scikit-learn modules you need inside each section.
# Example: from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42   # use this everywhere so your results are reproducible

# Dataset URLs (already provided for you)
TAXI_URL    = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/pu9kbeSaAtRZ7RxdJKX9_A/yellow-tripdata.csv"
OBESITY_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/GkDzb7bWrtvGXdPOfk6CIg/Obesity-level-prediction-dataset.csv"


---
# Section 1 — Supervised Learning: Regression
## Predicting taxi `tip_amount` (NYC Yellow Taxi)

Each row is a completed taxi trip. Your goal is to **predict the tip a passenger leaves**
(`tip_amount`, a continuous value) from the trip's characteristics. The available columns are:

`VendorID, passenger_count, trip_distance, RatecodeID, store_and_fwd_flag, PULocationID,
DOLocationID, payment_type, fare_amount, mta_tax, tolls_amount, improvement_surcharge, tip_amount`


### Part 1.1 — Load and explore the taxi data
Understand the shape, the data types, missing values, and the distribution of the target.

In [ ]:
# Load the taxi dataset from TAXI_URL into a DataFrame called `taxi`
taxi = pd.read_csv(TAXI_URL)

# Inspect the data
print("Taxi shape:", taxi.shape)
display(taxi.head())

print("\nTaxi info:")
taxi.info()

print("\nSummary statistics:")
display(taxi.describe())

print("\nMissing values per column:")
display(taxi.isna().sum())

# Visualise the distribution of the target `tip_amount`
plt.figure(figsize=(8, 5))
sns.histplot(data=taxi, x="tip_amount", bins=50, kde=True)
plt.title("Distribution of Taxi Tip Amounts")
plt.xlabel("Tip amount")
plt.ylabel("Number of trips")
plt.show()

# Check for impossible or suspicious values
print("Negative tips:", (taxi["tip_amount"] < 0).sum())
print("Zero or negative trip distances:", (taxi["trip_distance"] <= 0).sum())
print("Zero or negative fares:", (taxi["fare_amount"] <= 0).sum())


**Student Reasoning — Taxi data exploration**

> **Answer:** The data shape, first rows, summary statistics, and missing-value counts are printed above. I checked missing values because models cannot train properly on `NaN` values unless they are handled first. I also checked for impossible values such as negative tips, zero or negative trip distances, and zero or negative fares. The `tip_amount` distribution is usually right-skewed: many trips have small tips or zero tips, while a few trips have much larger tips. Because of this skew and the possible outliers, I cleaned invalid rows before modelling and used RMSE together with R² so I could judge both prediction error and overall fit.


### Part 1.2 — Preprocessing & feature engineering
Clean the data and create features that help predict the tip.

In [ ]:
# Handle missing / invalid rows
taxi_cleaned = taxi.copy()
taxi_cleaned = taxi_cleaned.dropna()

# Keep only realistic rows for this prediction task.
# I keep zero tips because "no tip" is a valid passenger outcome, but I remove negative tips.
taxi_cleaned = taxi_cleaned[
    (taxi_cleaned["trip_distance"] > 0) &
    (taxi_cleaned["fare_amount"] > 0) &
    (taxi_cleaned["tip_amount"] >= 0)
].copy()

# Feature engineering
# Do not use tip_amount to build features because that would leak the target.
taxi_cleaned["fare_per_mile"] = taxi_cleaned["fare_amount"] / taxi_cleaned["trip_distance"]
taxi_cleaned["total_surcharges"] = (
    taxi_cleaned["mta_tax"] +
    taxi_cleaned["tolls_amount"] +
    taxi_cleaned["improvement_surcharge"]
)

# Remove any infinite values created by division and drop remaining missing values if any.
taxi_cleaned = taxi_cleaned.replace([np.inf, -np.inf], np.nan).dropna()

# Separate feature types
taxi_numeric_cols = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "fare_per_mile",
    "total_surcharges"
]

taxi_categorical_cols = [
    "VendorID",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type"
]

# Encode categoricals with one-hot encoding.
# Scaling is done later in Part 1.3 and the scaler is fit only on the training set.
X_taxi = pd.get_dummies(
    taxi_cleaned[taxi_numeric_cols + taxi_categorical_cols],
    columns=taxi_categorical_cols,
    drop_first=True
)

y_taxi = taxi_cleaned["tip_amount"]

print("Cleaned taxi shape:", taxi_cleaned.shape)
print("Feature matrix shape after encoding:", X_taxi.shape)
display(X_taxi.head())


**Student Reasoning — Taxi preprocessing**

> **Answer:** I dropped missing rows because the dataset is large enough that removing a small number of incomplete rows is simpler than guessing replacement values. I removed trips with zero or negative distance, zero or negative fare, and negative tips because those values do not make sense for this prediction task. I kept zero tips because a passenger leaving no tip is a valid outcome. I engineered `fare_per_mile` because the same fare can mean different things depending on distance, and I engineered `total_surcharges` because extra charges may influence the final amount a passenger is reacting to. I used one-hot encoding for categorical columns so the model would not treat category IDs as ordered numbers. I used `StandardScaler` later because features like distance, fare, and surcharge values are on different scales.


### Part 1.3 — Train / Validation / Test split
A three-way split lets you tune on validation data and keep the test set as a final, unbiased check.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into train / validation / test using 60% / 20% / 20%.
# First, split off the test set.
X_taxi_train_val, X_taxi_test, y_taxi_train_val, y_taxi_test = train_test_split(
    X_taxi,
    y_taxi,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Then split the remaining 80% into 60% train and 20% validation.
# 0.25 of the remaining 80% = 20% of the full dataset.
X_taxi_train, X_taxi_val, y_taxi_train, y_taxi_val = train_test_split(
    X_taxi_train_val,
    y_taxi_train_val,
    test_size=0.25,
    random_state=RANDOM_STATE
)

# Fit the scaler on TRAIN only, then transform train, validation, and test.
taxi_scaler = StandardScaler()

X_taxi_train_scaled = X_taxi_train.copy()
X_taxi_val_scaled = X_taxi_val.copy()
X_taxi_test_scaled = X_taxi_test.copy()

X_taxi_train_scaled[taxi_numeric_cols] = taxi_scaler.fit_transform(X_taxi_train[taxi_numeric_cols])
X_taxi_val_scaled[taxi_numeric_cols] = taxi_scaler.transform(X_taxi_val[taxi_numeric_cols])
X_taxi_test_scaled[taxi_numeric_cols] = taxi_scaler.transform(X_taxi_test[taxi_numeric_cols])

print("Train shape:", X_taxi_train_scaled.shape)
print("Validation shape:", X_taxi_val_scaled.shape)
print("Test shape:", X_taxi_test_scaled.shape)


**Student Reasoning — Splitting**

> **Answer:** I used a 60/20/20 split: 60% of the rows for training, 20% for validation, and 20% for final testing. The validation set is useful because it lets me compare models and tune choices without touching the test set. The test set should only be used at the end, so it gives a more honest estimate of how well the final model performs on unseen data. The scaler must be fit only on the training data because fitting it on validation or test data would leak information from the future evaluation sets into the training process.


### Part 1.4 — Train a regressor and check for overfitting
Train a model and evaluate it on **train, validation, and test** sets.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Train two regression models: one simple baseline and one more flexible model.
regression_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=80,
        max_depth=12,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

def regression_scores(model, X, y):
    preds = model.predict(X)
    rmse = mean_squared_error(y, preds) ** 0.5
    r2 = r2_score(y, preds)
    return rmse, r2, preds

regression_results = []

for model_name, model in regression_models.items():
    model.fit(X_taxi_train_scaled, y_taxi_train)

    train_rmse, train_r2, _ = regression_scores(model, X_taxi_train_scaled, y_taxi_train)
    val_rmse, val_r2, _ = regression_scores(model, X_taxi_val_scaled, y_taxi_val)
    test_rmse, test_r2, _ = regression_scores(model, X_taxi_test_scaled, y_taxi_test)

    regression_results.append({
        "model": model_name,
        "train_RMSE": train_rmse,
        "train_R2": train_r2,
        "val_RMSE": val_rmse,
        "val_R2": val_r2,
        "test_RMSE": test_rmse,
        "test_R2": test_r2
    })

regression_results_df = pd.DataFrame(regression_results).sort_values("val_RMSE")
display(regression_results_df)

best_regression_model_name = regression_results_df.iloc[0]["model"]
best_regression_model = regression_models[best_regression_model_name]

print("Best model based on validation RMSE:", best_regression_model_name)

# Predict on test set with the best validation model
y_taxi_test_pred = best_regression_model.predict(X_taxi_test_scaled)

# Plot predicted vs actual tip for the test set
plot_sample = pd.DataFrame({
    "actual_tip": y_taxi_test,
    "predicted_tip": y_taxi_test_pred
}).sample(n=min(2000, len(y_taxi_test)), random_state=RANDOM_STATE)

plt.figure(figsize=(7, 6))
plt.scatter(plot_sample["actual_tip"], plot_sample["predicted_tip"], alpha=0.4)
max_tip = max(plot_sample["actual_tip"].max(), plot_sample["predicted_tip"].max())
plt.plot([0, max_tip], [0, max_tip], linestyle="--")
plt.title(f"Predicted vs Actual Taxi Tips ({best_regression_model_name})")
plt.xlabel("Actual tip amount")
plt.ylabel("Predicted tip amount")
plt.show()


**Student Reasoning — Regression evaluation & overfitting**

> **Answer:** I compared Linear Regression with a Random Forest Regressor and selected the model with the lowest validation RMSE. The validation results are shown in the metrics table above. Linear Regression is a useful baseline because it is simple and interpretable, while Random Forest is more flexible and can capture non-linear relationships. To judge overfitting, I compared the train, validation, and test RMSE/R² values. If the training score is much better than the validation and test scores, the model is overfitting. If all three scores are poor, the model is underfitting. If the three scores are close and reasonably strong, the model is well-fitted. If I saw overfitting, I would reduce the Random Forest depth, increase `min_samples_leaf`, use fewer features, or collect more data.


---
# Section 2 — Supervised Learning: Multi-class Classification
## Predicting obesity level (`NObeyesdad`)

Each row describes a person's eating habits and physical condition. Predict their
**obesity category** `NObeyesdad`, which has **7 classes**: `Insufficient_Weight, Normal_Weight,
Overweight_Level_I, Overweight_Level_II, Obesity_Type_I, Obesity_Type_II, Obesity_Type_III`.

Feature columns: `Gender, Age, Height, Weight, family_history_with_overweight, FAVC, FCVC, NCP,
CAEC, SMOKE, CH2O, SCC, FAF, TUE, CALC, MTRANS`.


### Part 2.1 — Load and explore the obesity data
Look at the shape, dtypes, missing values, and especially the **class balance** of the target.

In [ ]:
# Load the obesity dataset from OBESITY_URL into a DataFrame called `obesity`
obesity = pd.read_csv(OBESITY_URL)

# Inspect the data
print("Obesity shape:", obesity.shape)
display(obesity.head())

print("\nObesity info:")
obesity.info()

print("\nSummary statistics:")
display(obesity.describe())

print("\nMissing values per column:")
display(obesity.isna().sum())

# Show the class distribution of NObeyesdad
print("\nClass distribution:")
display(obesity["NObeyesdad"].value_counts())

plt.figure(figsize=(10, 5))
sns.countplot(data=obesity, x="NObeyesdad", order=obesity["NObeyesdad"].value_counts().index)
plt.title("Distribution of Obesity Classes")
plt.xlabel("Obesity level")
plt.ylabel("Number of people")
plt.xticks(rotation=45, ha="right")
plt.show()


**Student Reasoning — Obesity data exploration**

> **Answer:** The obesity dataset shape, column types, and missing-value counts are printed above. The numeric columns include variables such as `Age`, `Height`, `Weight`, `FCVC`, `NCP`, `CH2O`, `FAF`, and `TUE`. The categorical columns include `Gender`, `family_history_with_overweight`, `FAVC`, `CAEC`, `SMOKE`, `SCC`, `CALC`, and `MTRANS`. I checked the class distribution of `NObeyesdad` because classification models can become biased toward classes with more examples. If the classes are imbalanced, accuracy alone can be misleading, so macro-F1 is also useful because it gives each class equal importance.


### Part 2.2 — Preprocessing & feature engineering
Models need numeric input. Encode categoricals, scale numerics, and optionally add a feature.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Clean data
obesity_cleaned = obesity.copy().dropna()

# Feature engineering: BMI
# BMI is strongly related to obesity level, so it may make the task easier.
# I include it because it is a meaningful health feature, but I discuss the fairness issue in the reasoning.
obesity_cleaned["BMI"] = obesity_cleaned["Weight"] / (obesity_cleaned["Height"] ** 2)

# Separate target from features
X_obesity_raw = obesity_cleaned.drop(columns=["NObeyesdad"]).copy()
y_obesity_raw = obesity_cleaned["NObeyesdad"].copy()

# Binary yes/no columns -> 0/1
obesity_binary_cols = ["family_history_with_overweight", "FAVC", "SMOKE", "SCC"]
yes_no_map = {"yes": 1, "no": 0, "Yes": 1, "No": 0}

for col in obesity_binary_cols:
    X_obesity_raw[col] = X_obesity_raw[col].map(yes_no_map)

# Categorical columns -> one-hot encoding
obesity_categorical_cols = ["Gender", "CAEC", "CALC", "MTRANS"]

obesity_numeric_cols = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE",
    "BMI"
] + obesity_binary_cols

X_obesity = pd.get_dummies(
    X_obesity_raw[obesity_numeric_cols + obesity_categorical_cols],
    columns=obesity_categorical_cols,
    drop_first=False
)

# Encode target labels into integers
obesity_target_encoder = LabelEncoder()
y_obesity = obesity_target_encoder.fit_transform(y_obesity_raw)

print("Encoded obesity feature matrix shape:", X_obesity.shape)
print("Target classes:")
display(pd.DataFrame({
    "class_number": range(len(obesity_target_encoder.classes_)),
    "class_name": obesity_target_encoder.classes_
}))
display(X_obesity.head())


**Student Reasoning — Obesity preprocessing**

> **Answer:** I converted binary yes/no columns into 0/1 because those variables naturally have two possible states. I used one-hot encoding for columns such as `Gender`, `CAEC`, `CALC`, and `MTRANS` because I did not want the model to assume a false numerical order between categories. I engineered BMI using `Weight / Height²` because obesity level is closely related to body mass relative to height. However, BMI can make the task easier because the target categories are strongly connected to body-size measures, so the model may rely heavily on BMI and weight. I used `StandardScaler` for numeric variables so that variables measured on different scales would be comparable, especially for distance-based or linear models.


### Part 2.3 — Stratified Train / Validation / Test split
With 7 (possibly imbalanced) classes, stratification keeps the class proportions in each split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into train / validation / test using 60% / 20% / 20%.
# Stratify keeps the class proportions similar across all splits.
X_obesity_train_val, X_obesity_test, y_obesity_train_val, y_obesity_test = train_test_split(
    X_obesity,
    y_obesity,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_obesity
)

X_obesity_train, X_obesity_val, y_obesity_train, y_obesity_val = train_test_split(
    X_obesity_train_val,
    y_obesity_train_val,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_obesity_train_val
)

# Fit scaler on training data only
obesity_scaler = StandardScaler()

X_obesity_train_scaled = X_obesity_train.copy()
X_obesity_val_scaled = X_obesity_val.copy()
X_obesity_test_scaled = X_obesity_test.copy()

X_obesity_train_scaled[obesity_numeric_cols] = obesity_scaler.fit_transform(
    X_obesity_train[obesity_numeric_cols]
)
X_obesity_val_scaled[obesity_numeric_cols] = obesity_scaler.transform(
    X_obesity_val[obesity_numeric_cols]
)
X_obesity_test_scaled[obesity_numeric_cols] = obesity_scaler.transform(
    X_obesity_test[obesity_numeric_cols]
)

# For unsupervised clustering later, create one scaled matrix using all features but no target labels.
# This scaler is only for clustering and does not use the target variable.
obesity_cluster_scaler = StandardScaler()
X_obesity_scaled_all = X_obesity.copy()
X_obesity_scaled_all[obesity_numeric_cols] = obesity_cluster_scaler.fit_transform(
    X_obesity[obesity_numeric_cols]
)

print("Train shape:", X_obesity_train_scaled.shape)
print("Validation shape:", X_obesity_val_scaled.shape)
print("Test shape:", X_obesity_test_scaled.shape)

print("\nClass counts in train:")
display(pd.Series(y_obesity_train).value_counts().sort_index())

print("\nClass counts in validation:")
display(pd.Series(y_obesity_val).value_counts().sort_index())

print("\nClass counts in test:")
display(pd.Series(y_obesity_test).value_counts().sort_index())


**Student Reasoning — Splitting**

> **Answer:** I used a 60/20/20 split so the model has enough data to learn from, while still leaving separate validation and test sets. I used `stratify=y` because there are seven target classes, and I want each split to contain roughly the same proportion of every obesity level. Without stratification, some classes might be underrepresented or even missing in the validation or test set, which would make the evaluation unfair and unstable.


### Part 2.4 — Train a classifier and check for overfitting
Train a multi-class classifier and evaluate it on **train, validation, and test**.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

# Train one linear baseline and one more flexible classifier.
classification_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

def classification_scores(model, X, y):
    preds = model.predict(X)
    accuracy = accuracy_score(y, preds)
    macro_f1 = f1_score(y, preds, average="macro")
    return accuracy, macro_f1, preds

classification_results = []

for model_name, model in classification_models.items():
    model.fit(X_obesity_train_scaled, y_obesity_train)

    train_acc, train_f1, _ = classification_scores(model, X_obesity_train_scaled, y_obesity_train)
    val_acc, val_f1, _ = classification_scores(model, X_obesity_val_scaled, y_obesity_val)
    test_acc, test_f1, _ = classification_scores(model, X_obesity_test_scaled, y_obesity_test)

    classification_results.append({
        "model": model_name,
        "train_accuracy": train_acc,
        "train_macro_F1": train_f1,
        "val_accuracy": val_acc,
        "val_macro_F1": val_f1,
        "test_accuracy": test_acc,
        "test_macro_F1": test_f1
    })

classification_results_df = pd.DataFrame(classification_results).sort_values("val_macro_F1", ascending=False)
display(classification_results_df)

best_classifier_name = classification_results_df.iloc[0]["model"]
best_classifier = classification_models[best_classifier_name]

print("Best classifier based on validation macro-F1:", best_classifier_name)

# Predict on test set with the best validation model
y_obesity_test_pred = best_classifier.predict(X_obesity_test_scaled)

print("\nClassification report for best model on test set:")
print(classification_report(
    y_obesity_test,
    y_obesity_test_pred,
    target_names=obesity_target_encoder.classes_,
    zero_division=0
))

# Confusion matrix for the test set
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_obesity_test,
    y_obesity_test_pred,
    display_labels=obesity_target_encoder.classes_,
    xticks_rotation=45,
    ax=ax
)
plt.title(f"Confusion Matrix ({best_classifier_name})")
plt.show()


**Student Reasoning — Classification evaluation & overfitting**

> **Answer:** I compared Logistic Regression with a Random Forest Classifier. Logistic Regression is a good baseline for multi-class classification, while Random Forest can capture more complex relationships between lifestyle/body features and obesity level. I selected the model with the highest validation macro-F1 because macro-F1 treats all seven classes more equally than accuracy. To check overfitting, I compared the train, validation, and test accuracy/F1 values in the table above. A much higher train score than validation/test would mean overfitting. Similar validation and test scores suggest the model generalises reasonably well. From the confusion matrix, the hardest classes to separate are usually neighbouring weight categories, such as normal vs overweight or overweight level I vs level II, because their feature values can be close to each other.


---
# Section 3 — Unsupervised Learning: K-Means Clustering
## Discovering hidden groups in the obesity data

Now **pretend you never had the `NObeyesdad` labels.** Using only the *scaled feature matrix*
from Section 2, use **K-Means** to see whether people naturally fall into distinct profiles —
and then compare those clusters to the real obesity levels.


### Part 3.1 — Choose k, fit K-Means, and visualise


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Use ONLY the scaled obesity features; do not include the target labels.
X_cluster = X_obesity_scaled_all.copy()

# Elbow method and silhouette score for k = 2..10
k_values = range(2, 11)
inertias = []
silhouettes = []

for k in k_values:
    kmeans_temp = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10
    )
    temp_labels = kmeans_temp.fit_predict(X_cluster)
    inertias.append(kmeans_temp.inertia_)
    silhouettes.append(
        silhouette_score(
            X_cluster,
            temp_labels,
            sample_size=min(1000, len(X_cluster)),
            random_state=RANDOM_STATE
        )
    )

k_selection_df = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette": silhouettes
})

display(k_selection_df)

plt.figure(figsize=(7, 5))
plt.plot(k_selection_df["k"], k_selection_df["inertia"], marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(k_selection_df["k"], k_selection_df["silhouette"], marker="o")
plt.title("Silhouette Score by k")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.show()

# Choose k using the highest silhouette score.
# If your elbow plot clearly suggests another k, you can manually change chosen_k.
chosen_k = int(k_selection_df.loc[k_selection_df["silhouette"].idxmax(), "k"])
print("Chosen k:", chosen_k)

kmeans = KMeans(
    n_clusters=chosen_k,
    random_state=RANDOM_STATE,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X_cluster)

# Visualise clusters in 2D using PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
cluster_points_2d = pca.fit_transform(X_cluster)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    cluster_points_2d[:, 0],
    cluster_points_2d[:, 1],
    c=cluster_labels,
    alpha=0.7
)
plt.title("K-Means Clusters Visualised with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="Cluster")
plt.show()

print("Explained variance by the two PCA components:", pca.explained_variance_ratio_)


In [ ]:
# Compare clusters with the true obesity levels.
# The true labels are used only AFTER clustering to interpret the clusters.
obesity_clustered = obesity_cleaned.copy()
obesity_clustered["cluster"] = cluster_labels

cluster_vs_obesity_counts = pd.crosstab(
    obesity_clustered["cluster"],
    obesity_clustered["NObeyesdad"]
)

cluster_vs_obesity_percent = pd.crosstab(
    obesity_clustered["cluster"],
    obesity_clustered["NObeyesdad"],
    normalize="index"
) * 100

print("Cluster vs true obesity level — counts:")
display(cluster_vs_obesity_counts)

print("Cluster vs true obesity level — row percentages:")
display(cluster_vs_obesity_percent.round(1))


**Student Reasoning — Clustering**

> **Answer:** I used the Elbow method and silhouette score to choose `k`. Inertia always decreases as `k` increases, so I looked for a point where the decrease started to slow down. I also checked the silhouette score because it measures how separated the clusters are. I then used the selected `k` to fit K-Means and compared the cluster labels with the real `NObeyesdad` labels using a crosstab. If one cluster has a high percentage of a specific obesity level, that means the unsupervised grouping partly matches the real label. If a cluster contains many different obesity levels, then K-Means has found broader patterns rather than the exact labelled categories. In a public-health setting, these clusters could help identify groups with similar lifestyle or physical profiles, especially when full medical labels are expensive or unavailable.


---
# Section 4 — Reflection

*Answer in a few sentences each:*

1. **Supervised vs unsupervised:** The supervised classifier learned a direct mapping from the input features to the known obesity labels. K-Means did not know the labels, so it could only group people based on feature similarity. This means the classifier is better for predicting a known category, while K-Means is useful for discovering hidden patterns or patient profiles.

2. **Regression vs classification:** In regression, the taxi model predicted a continuous value, so I evaluated it with RMSE and R². RMSE tells me the typical size of the prediction error, while R² tells me how much variation the model explains. In classification, the obesity model predicted one of seven categories, so I used accuracy, macro-F1, a classification report, and a confusion matrix.

3. **Overfitting:** I checked overfitting by comparing train, validation, and test scores. The biggest warning sign would be a model that performs much better on training data than on validation/test data. The most effective way to reduce that would be to limit model complexity, such as reducing tree depth, increasing `min_samples_leaf`, or choosing a simpler model.


---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel → Restart & Run All`).
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] Plots are visible in the saved notebook.
- [ ] Notebook committed and pushed to your `lab-2-predictive-analytics` repository.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository

---
#### Grading guide (100 pts)
| Area | Pts |
|------|-----|
| Section 1 — Regression (load, preprocess, split, model, overfitting) | 30 |
| Section 2 — Classification (load, preprocess, stratified split, model, overfitting) | 30 |
| Section 3 — K-Means clustering (k selection, fit, visualise, compare) | 20 |
| Reasoning boxes & Section 4 reflection | 15 |
| Reproducibility (runs clean, random_state, tidy code) | 5 |
